<a href="https://colab.research.google.com/github/mlbyhoang/marketing-attribution/blob/main/notebooks/11_model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11 – Model Evaluation & Comparison

This notebook evaluates and compares the Logistic Regression baseline and three GRU architectures:

- Single-Layer GRU
- GRU with Touchpoint Attention
- Dual-Attention GRU (DAGRU)

The evaluation focuses on predictive performance using PR-AUC, ROC-AUC, precision, recall, F1-score, and confusion matrices.

In [ ]:
# Import libraries
import os
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive, userdata

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    log_loss
)

In [ ]:
# Mount Google Drive
drive.mount("/content/drive")

# Load results directory from Colab Secrets
save_path = userdata.get("MTA_RESULTS_PATH")

# Verify that the results directory exists
if not os.path.isdir(save_path):
    raise FileNotFoundError(
        f"Results directory not found: {save_path}"
    )

print(f"Results directory: {save_path}")

In [ ]:
# Load evaluation results for all models
model_files = {
    "LR + ADASYN (Baseline)": "results_lr.json",
    "Single-Layer GRU": "results_ablation_1.json",
    "GRU with Touchpoint Attention": "results_ablation_2.json",
    "Dual-Attention GRU (DAGRU)": "results_ablation_3.json"
}

results = []

for model_name, filename in model_files.items():
    file_path = os.path.join(save_path, filename)

    with open(file_path, "r") as f:
        metrics = json.load(f)

    # Support current and historical metric names
    pr_auc = metrics.get("pr_auc", metrics.get("pr_auc_conv"))
    roc_auc = metrics.get("roc_auc", metrics.get("roc_auc_conv"))

    if pr_auc is None or roc_auc is None:
        raise KeyError(
            f"Missing evaluation metrics for {model_name}"
        )

    results.append({
        "Model": model_name,
        "PR-AUC": pr_auc,
        "ROC-AUC": roc_auc
    })

# Create comparison table
df_results = pd.DataFrame(results)

# Calculate incremental PR-AUC differences
df_results["Delta PR-AUC"] = df_results["PR-AUC"].diff()

print("===== Model Comparison =====")
display(df_results.round(4))

# Calculate overall improvement relative to the baseline
baseline = df_results.iloc[0]
final_model = df_results.iloc[-1]

print("\nOverall improvement: LR -> DAGRU")
print(
    f"PR-AUC: "
    f"{final_model['PR-AUC'] - baseline['PR-AUC']:+.4f}"
)
print(
    f"ROC-AUC: "
    f"{final_model['ROC-AUC'] - baseline['ROC-AUC']:+.4f}"
)

Mounted at /content/drive


In [ ]:
# Load predicted conversion probabilities
prediction_files = {
    "LR + ADASYN (Baseline)": "y_prob_lr.npy",
    "Single-Layer GRU": "y_prob_ablation_1.npy",
    "GRU with Touchpoint Attention": "y_prob_ablation_2.npy",
    "Dual-Attention GRU (DAGRU)": "y_prob_ablation_3.npy"
}

predictions = {}

for model_name, filename in prediction_files.items():
    file_path = os.path.join(save_path, filename)
    predictions[model_name] = np.load(file_path).flatten()

# Load test labels
y_test_arr = np.load(
    os.path.join(save_path, "y_test.npy")
).flatten()

# Validate test labels
if not np.isin(y_test_arr, [0, 1]).all():
    raise ValueError("Test labels must be binary.")

if np.unique(y_test_arr).size != 2:
    raise ValueError("Both classes must be present in the test set.")

print(f"Test samples: {len(y_test_arr)}")
print(f"Conversions: {int(np.sum(y_test_arr))}")
print(f"Conversion rate: {np.mean(y_test_arr):.2%}")

# Validate prediction arrays
for model_name, y_prob in predictions.items():
    if len(y_prob) != len(y_test_arr):
        raise ValueError(
            f"Prediction length mismatch for {model_name}"
        )

    if not np.all(np.isfinite(y_prob)):
        raise ValueError(
            f"Invalid prediction values for {model_name}"
        )

    if np.any((y_prob < 0) | (y_prob > 1)):
        raise ValueError(
            f"Probabilities outside [0, 1] for {model_name}"
        )

    print(f"{model_name}: {len(y_prob)} predictions")

print("\nAll prediction arrays passed the basic checks.")


In [ ]:

# Validate saved evaluation metrics
validation_results = []

for model_name, y_prob in predictions.items():

    # Recalculate metrics from saved predictions
    calculated_pr_auc = average_precision_score(
        y_test_arr, y_prob
    )

    calculated_roc_auc = roc_auc_score(
        y_test_arr, y_prob
    )

    # Retrieve saved metrics
    saved_metrics = df_results.loc[
        df_results["Model"] == model_name
    ].iloc[0]

    saved_pr_auc = saved_metrics["PR-AUC"]
    saved_roc_auc = saved_metrics["ROC-AUC"]

    # Compare saved and recalculated metrics
    pr_match = np.isclose(
        calculated_pr_auc,
        saved_pr_auc,
        atol=1e-4,
        rtol=0
    )

    roc_match = np.isclose(
        calculated_roc_auc,
        saved_roc_auc,
        atol=1e-4,
        rtol=0
    )

    validation_results.append({
        "Model": model_name,
        "Saved PR-AUC": saved_pr_auc,
        "Calculated PR-AUC": calculated_pr_auc,
        "Saved ROC-AUC": saved_roc_auc,
        "Calculated ROC-AUC": calculated_roc_auc,
        "Metrics Match": bool(pr_match and roc_match)
    })

# Create validation table
df_validation = pd.DataFrame(validation_results)

print("===== Evaluation Metrics Validation =====")
display(df_validation.round(4))

# Check overall consistency
if not df_validation["Metrics Match"].all():
    raise ValueError(
        "Saved metrics do not match recalculated metrics."
    )

print("All saved metrics match the recalculated values.")

In [ ]:
# Calculate cross-entropy loss for all models
ce_losses = {
    model_name: log_loss(y_test_arr, y_prob)
    for model_name, y_prob in predictions.items()
}

# Add cross-entropy loss to the comparison table
df_results["CE-Loss"] = df_results["Model"].map(ce_losses)

print("===== Model Comparison with Cross-Entropy Loss =====")
display(df_results.round(4))

# Save updated comparison table
comparison_path = os.path.join(
    save_path,
    "final_model_comparison.csv"
)

df_results.to_csv(comparison_path, index=False)

print("Model comparison saved.")


In [ ]:
# Visualize cross-entropy loss across models
short_labels = [
    "LR",
    "GRU",
    "GRU + Attention",
    "DAGRU"
]

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    short_labels,
    df_results["CE-Loss"],
    alpha=0.85
)

# Display loss values above bars
ax.bar_label(
    bars,
    fmt="%.4f",
    padding=3
)

ax.set_ylabel("Binary Cross-Entropy Loss")
ax.set_title("Test Cross-Entropy Loss by Model")
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3, axis="y")
ax.set_axisbelow(True)

plt.tight_layout()

# Save figure
figure_path = os.path.join(
    save_path,
    "ce_loss_comparison.png"
)

plt.savefig(
    figure_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("Cross-entropy loss comparison saved.")

In [ ]:
# Compare Precision-Recall and ROC curves across models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_name, y_prob in predictions.items():

    # Calculate Precision-Recall curve
    precision, recall, _ = precision_recall_curve(
        y_test_arr, y_prob
    )

    pr_auc = average_precision_score(
        y_test_arr, y_prob
    )

    axes[0].plot(
        recall,
        precision,
        label=f"{model_name} (AP={pr_auc:.3f})"
    )

    # Calculate ROC curve
    fpr, tpr, _ = roc_curve(
        y_test_arr, y_prob
    )

    roc_auc = roc_auc_score(
        y_test_arr, y_prob
    )

    axes[1].plot(
        fpr,
        tpr,
        label=f"{model_name} (AUC={roc_auc:.3f})"
    )

# Precision-Recall baseline
conversion_rate = np.mean(y_test_arr)

axes[0].axhline(
    conversion_rate,
    linestyle="--",
    color="gray",
    label=f"Random baseline ({conversion_rate:.3f})"
)

axes[0].set_title("Precision-Recall Curves")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# ROC baseline
axes[1].plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    label="Random baseline"
)

axes[1].set_title("ROC Curves")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()

# Save figure
figure_path = os.path.join(
    save_path,
    "ablation_final_comparison.png"
)

plt.savefig(
    figure_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("Precision-Recall and ROC comparison saved.")


In [ ]:
# Load training histories for all GRU models
history_files = {
    "Single-Layer GRU": "history1.pkl",
    "GRU with Touchpoint Attention": "history2.pkl",
    "Dual-Attention GRU (DAGRU)": "history3.pkl"
}

histories = {}

for model_name, filename in history_files.items():
    file_path = os.path.join(save_path, filename)

    with open(file_path, "rb") as f:
        history = pickle.load(f)

    # Support both Keras History objects and dictionaries
    if hasattr(history, "history"):
        history = history.history

    histories[model_name] = history

# Plot training and validation loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_name, history in histories.items():

    if "loss" not in history or "val_loss" not in history:
        raise KeyError(
            f"Missing training or validation loss for {model_name}"
        )

    epochs = range(1, len(history["loss"]) + 1)
    val_epochs = range(1, len(history["val_loss"]) + 1)

    axes[0].plot(
        epochs,
        history["loss"],
        label=model_name
    )

    axes[1].plot(
        val_epochs,
        history["val_loss"],
        label=model_name
    )

axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].set_title("Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()

# Save figure
figure_path = os.path.join(
    save_path,
    "loss_curves.png"
)

plt.savefig(
    figure_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("Training history comparison saved.")


In [ ]:
# Load DAGRU evaluation results
dagru_results_path = os.path.join(
    save_path,
    "results_ablation_3.json"
)

with open(dagru_results_path, "r") as f:
    dagru_results = json.load(f)

# Extract available auxiliary click metrics
click_metric_keys = [
    "pr_auc_click",
    "roc_auc_click",
    "click_rate",
    "click_weight"
]

click_metrics = {
    key: dagru_results[key]
    for key in click_metric_keys
    if key in dagru_results
}

print("===== DAGRU Auxiliary Click Metrics =====")

if click_metrics:
    df_click_metrics = pd.DataFrame(
        click_metrics.items(),
        columns=["Metric", "Value"]
    )

    display(df_click_metrics.round(4))
else:
    print("No auxiliary click metrics found in saved results.")


In [ ]:
# Check availability of validation predictions
validation_files = {
    "LR + ADASYN (Baseline)": "y_prob_lr_val.npy",
    "Single-Layer GRU": "y_prob_ablation_1_val.npy",
    "GRU with Touchpoint Attention": "y_prob_ablation_2_val.npy",
    "Dual-Attention GRU (DAGRU)": "y_prob_ablation_3_val.npy"
}

validation_labels_path = os.path.join(
    save_path,
    "y_val.npy"
)

required_files = [
    validation_labels_path,
    *[
        os.path.join(save_path, filename)
        for filename in validation_files.values()
    ]
]

missing_files = [
    path for path in required_files
    if not os.path.isfile(path)
]

validation_available = len(missing_files) == 0

if not validation_available:
    print("Validation predictions are not fully available.")
    print("Threshold optimization will be skipped.")

    for path in missing_files:
        print(f"Missing: {os.path.basename(path)}")

else:
    # Load validation labels
    y_val_arr = np.load(validation_labels_path).flatten()

    if not np.isin(y_val_arr, [0, 1]).all():
        raise ValueError("Validation labels must be binary.")

    if np.unique(y_val_arr).size != 2:
        raise ValueError(
            "Both classes must be present in the validation set."
        )

    # Load and validate prediction probabilities
    validation_predictions = {}

    for model_name, filename in validation_files.items():
        file_path = os.path.join(save_path, filename)
        y_prob_val = np.load(file_path).flatten()

        if len(y_prob_val) != len(y_val_arr):
            raise ValueError(
                f"Validation length mismatch for {model_name}"
            )

        if not np.all(np.isfinite(y_prob_val)):
            raise ValueError(
                f"Invalid validation predictions for {model_name}"
            )

        if np.any((y_prob_val < 0) | (y_prob_val > 1)):
            raise ValueError(
                f"Probabilities outside [0, 1] for {model_name}"
            )

        validation_predictions[model_name] = y_prob_val

    print("All validation prediction arrays passed the basic checks.")
    print(f"Validation samples: {len(y_val_arr)}")
    print(f"Conversion rate: {np.mean(y_val_arr):.2%}")

In [ ]:
# Optimize classification thresholds using validation data
if validation_available:

    threshold_results = []

    for model_name, y_prob_val in validation_predictions.items():

        # Calculate precision and recall across thresholds
        precision, recall, thresholds = precision_recall_curve(
            y_val_arr, y_prob_val
        )

        # Calculate F1-score for each available threshold
        f1_scores = (
            2 * precision[:-1] * recall[:-1]
            / (precision[:-1] + recall[:-1] + 1e-10)
        )

        # Select threshold with the highest validation F1-score
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]

        # Apply selected threshold to test predictions
        y_pred_test = (
            predictions[model_name] >= best_threshold
        ).astype(int)

        threshold_results.append({
            "Model": model_name,
            "Threshold": best_threshold,
            "Precision": precision_score(
                y_test_arr, y_pred_test, zero_division=0
            ),
            "Recall": recall_score(
                y_test_arr, y_pred_test, zero_division=0
            ),
            "F1-Score": f1_score(
                y_test_arr, y_pred_test, zero_division=0
            )
        })

    # Create final classification comparison
    df_final = pd.DataFrame(threshold_results)

    print("===== Threshold-Optimized Test Performance =====")
    display(df_final.round(4))

else:
    print(
        "Threshold optimization skipped: "
        "validation predictions are unavailable."
    )


In [ ]:

# Evaluate classification performance on the test set
if validation_available:

    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    axes = axes.flatten()

    for ax, model_name in zip(axes, predictions.keys()):

        # Retrieve validation-optimized threshold
        threshold = df_final.loc[
            df_final["Model"] == model_name,
            "Threshold"
        ].iloc[0]

        # Convert test probabilities into binary predictions
        y_pred = (
            predictions[model_name] >= threshold
        ).astype(int)

        # Calculate confusion matrix
        cm = confusion_matrix(
            y_test_arr,
            y_pred,
            labels=[0, 1]
        )

        # Visualize confusion matrix
        im = ax.imshow(cm, cmap="Blues")

        ax.set_title(model_name)
        ax.set_xlabel("Predicted Label")
        ax.set_ylabel("True Label")

        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(["No Conversion", "Conversion"])
        ax.set_yticklabels(["No Conversion", "Conversion"])

        # Display counts inside matrix cells
        for i in range(2):
            for j in range(2):
                ax.text(
                    j,
                    i,
                    str(cm[i, j]),
                    ha="center",
                    va="center",
                    color=(
                        "white"
                        if cm[i, j] > cm.max() / 2
                        else "black"
                    )
                )

        print(f"\n===== {model_name} =====")
        print(f"Threshold: {threshold:.4f}")
        print(
            classification_report(
                y_test_arr,
                y_pred,
                target_names=["No Conversion", "Conversion"],
                zero_division=0
            )
        )

    plt.tight_layout()

    # Save confusion matrix comparison
    figure_path = os.path.join(
        save_path,
        "confusion_matrices.png"
    )

    plt.savefig(
        figure_path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

    print("Confusion matrix comparison saved.")

else:
    print(
        "Classification reports and confusion matrices skipped: "
        "validation predictions are unavailable."
    )

In [ ]:
# Save threshold-based classification results
if validation_available:

    output_path = os.path.join(
        save_path,
        "final_classification_results.csv"
    )

    df_final.to_csv(
        output_path,
        index=False
    )

    print("===== Final Classification Results =====")
    display(df_final.round(4))

    print("Classification results saved successfully.")

else:
    print(
        "Classification results not saved: "
        "validation predictions are unavailable."
    )